<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/LLM_model_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install transformers torch pandas numpy scikit-learn tqdm
!pip install -U bitsandbytes>=0.46.1

In [6]:
from huggingface_hub import HfApi, login, create_repo
import os
from huggingface_hub import notebook_login

notebook_login()

In [13]:
# ==========================================
# 0. 環境安裝 (若是第一次在 Colab 執行，請取消註解並執行)
# ==========================================
# !pip install -q transformers datasets peft bitsandbytes accelerate pandas numpy scikit-learn

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import f1_score
import datetime

def load_fold_data_from_github(fold_num):
    """從 GitHub Raw 連結動態讀取指定 Fold 的訓練與驗證資料。"""
    base_url = "https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/main/app/data/clean_data/"
    train_url = f"{base_url}train_fold_{fold_num}.csv"
    val_url = f"{base_url}val_fold_{fold_num}.csv"

    print(f"\n📥 正在從 GitHub 獲取 Fold {fold_num} 的資料...")
    try:
        train_df = pd.read_csv(train_url)
        val_df = pd.read_csv(val_url)
        print(f"✅ 獲取成功！訓練集: {len(train_df)} 筆 | 驗證集: {len(val_df)} 筆")
        return train_df, val_df
    except Exception as e:
        print(f"❌ 讀取失敗，請檢查網址或權限。錯誤: {e}")
        return None, None

# ==========================================
# 0. 全域設定與超參數
# ==========================================
# 修改點：更換為 TAIDE 模型
MODEL_NAME = "taide/TAIDE-LX-7B"
MAX_LEN = 512
# 修改點：為適應 8B 模型，Batch Size 下調至 4 以防 OOM
BATCH_SIZE = 4
EPOCHS = 5
LR = 2e-4 # 修改點：LoRA 建議的學習率通常較高 (原本 BERT 為 2e-5)
OUTPUT_DIR = "./folds_data/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. 資料前處理與 Dataset
# ==========================================
class ESG_MTL_Dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

        self.head_len = int((max_len - 2) * 0.25)
        self.tail_len = (max_len - 2) - self.head_len

        # 標籤映射
        self.t1_map = {"No": 0, "Yes": 1}
        self.t2_map = {"already": 0, "within_2_years": 1, "between_2_and_5_years": 2, "longer_than_5_years": 3, "more_than_5_years": 3}
        self.t3_map = {"No": 0, "Yes": 1}
        self.t4_map = {"Clear": 0, "Not Clear": 1, "Misleading": 2}

    def _process_esg_type(self, esg_val):
        if pd.isna(esg_val) or str(esg_val).strip() == "": return "[ESG_UNK]"
        return " ".join([f"[ESG_{t.strip()}]" for t in str(esg_val).split(';')])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        esg_prefix = self._process_esg_type(row.get('esg_type', ''))
        raw_text = str(row['data'])
        full_text = f"{esg_prefix} 文本內容：{raw_text}"

        tokens = self.tokenizer.encode(full_text, add_special_tokens=False)
        if len(tokens) > (self.max_len - 2):
            tokens = tokens[:self.head_len] + tokens[-self.tail_len:]

        # 修改點：適配 Llama 3 的 BOS/EOS token
        input_ids = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]

        # 修改點：強制左側填充 (Left Padding)，確保最後一個有效 token 永遠在索引 -1
        pad_len = self.max_len - len(input_ids)
        input_ids = [self.tokenizer.pad_token_id] * pad_len + input_ids
        attention_mask = [0] * pad_len + [1] * (self.max_len - pad_len)

        item = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long)
        }

        if self.is_test: return item

        item['t1_label'] = torch.tensor(self.t1_map.get(str(row.get('promise_status')), -1), dtype=torch.float)
        item['t2_label'] = torch.tensor(self.t2_map.get(str(row.get('verification_timeline')), -1), dtype=torch.long)
        item['t3_label'] = torch.tensor(self.t3_map.get(str(row.get('evidence_status')), -1), dtype=torch.float)
        item['t4_label'] = torch.tensor(self.t4_map.get(str(row.get('evidence_quality')), -1), dtype=torch.long)

        return item

# ==========================================
# 2. 統一多任務模型架構 (TAIDE LoRA Backbone + 4 Heads)
# ==========================================
class ESG_Unified_MTL_Model(nn.Module):
    # 修改點：不再內部初始化模型，而是接收已經設定好 LoRA 的 backbone
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        hidden_size = self.backbone.config.hidden_size # TAIDE 8B 通常是 4096

        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])

        # 分類頭參數稍微放大以對應 4096 維度的特徵
        self.t1_head = nn.Sequential(nn.Linear(hidden_size, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 1))
        self.t3_head = nn.Sequential(nn.Linear(hidden_size, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 1))
        self.t2_head = nn.Sequential(nn.Linear(hidden_size, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 4))
        self.t4_head = nn.Sequential(nn.Linear(hidden_size, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 3))

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        # 修改點：因為採用左側填充，序列最後一個位置 [-1] 必定是 EOS token 或最後一個文字 token
        cls_output = outputs.last_hidden_state[:, -1, :]

        t1_logits = self.t1_head(cls_output).squeeze(-1)
        t3_logits = self.t3_head(cls_output).squeeze(-1)
        t2_logits = self.t2_head(cls_output)
        t4_logits = self.t4_head(cls_output)

        return t1_logits, t2_logits, t3_logits, t4_logits

# ==========================================
# 3. 損失函數與屏蔽機制 (維持不變)
# ==========================================
def calculate_mtl_loss(preds, labels):
    t1_pred, t2_pred, t3_pred, t4_pred = preds
    t1_lbl, t2_lbl, t3_lbl, t4_lbl = labels

    bce_loss = nn.BCEWithLogitsLoss()
    loss_t1 = bce_loss(t1_pred, t1_lbl)

    valid_mask = (t1_lbl == 1)
    loss_t2 = loss_t3 = loss_t4 = torch.tensor(0.0, device=DEVICE)

    if valid_mask.sum() > 0:
        ce_loss_t2 = nn.CrossEntropyLoss(ignore_index=-1)
        loss_t2 = ce_loss_t2(t2_pred[valid_mask], t2_lbl[valid_mask])

        t3_valid = valid_mask & (t3_lbl != -1)
        if t3_valid.sum() > 0:
            loss_t3 = bce_loss(t3_pred[t3_valid], t3_lbl[t3_valid])

        t4_valid = valid_mask & (t4_lbl != -1)
        if t4_valid.sum() > 0:
            t4_weights = torch.tensor([1.0, 2.0, 5.0], device=DEVICE)
            ce_loss_t4 = nn.CrossEntropyLoss(weight=t4_weights, ignore_index=-1)
            loss_t4 = ce_loss_t4(t4_pred[t4_valid], t4_lbl[t4_valid])

    total_loss = 0.2 * loss_t1 + 0.15 * loss_t2 + 0.3 * loss_t3 + 0.35 * loss_t4
    return total_loss

# ==========================================
# 4. 訓練流程與推論管線
# ==========================================
def train_and_predict():
    print("🚀 初始化 Tokenizer...")
    # 修改點：使用 AutoTokenizer 載入 TAIDE Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # 確保有 pad_token，並將填充方向設為左邊
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    special_tokens = {'additional_special_tokens': ['[ESG_E]', '[ESG_S]', '[ESG_G]', '[ESG_UNK]']}
    tokenizer.add_special_tokens(special_tokens)

    # # 修改點：設定 4-Bit 量化配置以節省 VRAM
    # bnb_config = BitsAndBytesConfig(
    #     load_in_4bit=True,
    #     bnb_4bit_use_double_quant=True,
    #     bnb_4bit_quant_type="nf4",
    #     bnb_4bit_compute_dtype=torch.bfloat16
    # )

    # 修改點：設定 LoRA (PEFT) 配置
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="FEATURE_EXTRACTION"
    )

    for fold in range(1, 6):
        print(f"\n{'='*40}")
        print(f"🏆 開始訓練 Fold {fold}")
        print(f"{'='*40}")

        train_df, val_df = load_fold_data_from_github(fold)
        if train_df is None or val_df is None: continue

        train_dataset = ESG_MTL_Dataset(train_df, tokenizer, MAX_LEN)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_dataset = ESG_MTL_Dataset(val_df, tokenizer, MAX_LEN)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

        # 修改點：載入底層模型、套用量化與 LoRA
        print("載入 TAIDE 模型並注入 LoRA 參數...")
        base_model = AutoModel.from_pretrained(
            MODEL_NAME,
            device_map={'': torch.cuda.current_device()} # 分配至目前的 GPU
        )
        base_model.resize_token_embeddings(len(tokenizer))
        base_model = prepare_model_for_kbit_training(base_model)
        peft_backbone = get_peft_model(base_model, lora_config)

        # 實例化客製化的多任務模型
        model = ESG_Unified_MTL_Model(peft_backbone)
        model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = (batch['t1_label'].to(DEVICE), batch['t2_label'].to(DEVICE),
                          batch['t3_label'].to(DEVICE), batch['t4_label'].to(DEVICE))

                preds = model(input_ids, attention_mask)
                loss = calculate_mtl_loss(preds, labels)

                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            model.eval()
            val_total_loss = 0
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(DEVICE)
                    attention_mask = batch['attention_mask'].to(DEVICE)
                    labels = (batch['t1_label'].to(DEVICE), batch['t2_label'].to(DEVICE),
                              batch['t3_label'].to(DEVICE), batch['t4_label'].to(DEVICE))

                    preds = model(input_ids, attention_mask)
                    val_loss = calculate_mtl_loss(preds, labels)
                    val_total_loss += val_loss.item()

            print(f"  - Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_total_loss/len(val_loader):.4f}")

        # 儲存該 Fold 包含 LoRA 與 Classifier 的權重
        save_path = f"best_mtl_model_fold_{fold}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"💾 Fold {fold} 模型已儲存至 {save_path}")

        # --- 測試集推論區塊 (維持不變) ---
        print("\n🔮 準備執行新資料推論...")
        test_data = {
            'id': [99991, 99992],
            'esg_type': ['E', 'S;G'],
            'data': ['本公司承諾於 2030 年達成 100% 綠電使用，目前已導入太陽能版，預計每年減碳 10%。',
                     '我們致力於促進員工福祉，但具體計畫還在研議中。']
        }
        test_df = pd.DataFrame(test_data)
        test_dataset = ESG_MTL_Dataset(test_df, tokenizer, MAX_LEN, is_test=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        model.eval()
        results = []
        inv_t1, inv_t2, inv_t3, inv_t4 = {0: "No", 1: "Yes"}, {0: "already", 1: "within_2_years", 2: "between_2_and_5_years", 3: "longer_than_5_years"}, {0: "No", 1: "Yes"}, {0: "Clear", 1: "Not Clear", 2: "Misleading"}

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                t1_log, t2_log, t3_log, t4_log = model(input_ids, attention_mask)

                t1_preds = (torch.sigmoid(t1_log) > 0.5).long().cpu().numpy()
                t2_preds = torch.argmax(t2_log, dim=1).cpu().numpy()
                t3_preds = (torch.sigmoid(t3_log) > 0.5).long().cpu().numpy()
                t4_preds = torch.argmax(t4_log, dim=1).cpu().numpy()

                for i in range(len(t1_preds)):
                    if t1_preds[i] == 0:
                        results.append({"promise_status": "No", "verification_timeline": "N/A", "evidence_status": "N/A", "evidence_quality": "N/A"})
                    else:
                        t2_res = inv_t2[t2_preds[i]]
                        if t3_preds[i] == 0:
                            results.append({"promise_status": "Yes", "verification_timeline": t2_res, "evidence_status": "No", "evidence_quality": "N/A"})
                        else:
                            results.append({"promise_status": "Yes", "verification_timeline": t2_res, "evidence_status": "Yes", "evidence_quality": inv_t4[t4_preds[i]]})

        final_output = pd.concat([pd.DataFrame({'id': test_df['id']}), pd.DataFrame(results)], axis=1)
        print(final_output)

        del model, base_model, peft_backbone
        torch.cuda.empty_cache()

    print("\n🎉 5-Fold 訓練全數完成！你現在擁有了 5 個不同視角的專家模型。")

if __name__ == "__main__":
    train_and_predict()

🚀 初始化 Tokenizer...

🏆 開始訓練 Fold 1

📥 正在從 GitHub 獲取 Fold 1 的資料...
✅ 獲取成功！訓練集: 911 筆 | 驗證集: 200 筆
載入 TAIDE 模型並注入 LoRA 參數...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LlamaModel LOAD REPORT from: taide/TAIDE-LX-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 39.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.17 GiB is allocated by PyTorch, and 237.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# from huggingface_hub import HfApi, login, create_repo
# import os
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
# from huggingface_hub import HfApi
# from transformers import AutoTokenizer

# def push_models_to_hf(api, repo_id):
#     print(f"🚀 正在 Hugging Face 建立儲存庫: {repo_id}")
#     # 若 Repo 已存在則忽略錯誤
#     api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

#     # 1. 儲存並上傳特製的 Tokenizer (非常重要！因為我們加了 [ESG_E] 等 Tokens)
#     print("📦 正在封裝並上傳 Tokenizer...")
#     # tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')
#     tokenizer.add_special_tokens({'additional_special_tokens': ['[ESG_E]', '[ESG_S]', '[ESG_G]', '[ESG_UNK]']})

#     tokenizer_path = "./custom_tokenizer"
#     tokenizer.save_pretrained(tokenizer_path)
#     api.upload_folder(
#         folder_path=tokenizer_path,
#         repo_id=repo_id,
#         path_in_repo="tokenizer" # 在 Repo 中的資料夾名稱
#     )

#     # 2. 上傳 5 個 Fold 的模型權重
#     for fold in range(1, 6):
#         file_path = f"best_mtl_model_fold_{fold}.pth"
#         print(f"⬆️ 正在上傳 {file_path} ...")

#         try:
#             api.upload_file(
#                 path_or_fileobj=file_path,
#                 path_in_repo=file_path,
#                 repo_id=repo_id,
#                 repo_type="model"
#             )
#         except Exception as e:
#             print(f"❌ Fold {fold} 上傳失敗，請檢查檔案是否存在: {e}")

#     print(f"🎉 系統部署完成！你的 5-Fold 專家模型已永久保存在: https://huggingface.co/{repo_id}")



# # 執行上傳 (請替換為你的 HF 帳號 / 專案名稱)
# api = HfApi()
# username = api.whoami()['name']
# push_models_to_hf(api, f"{username}/VeriPromise_ESG_2026_9906(LLM)")